In [1]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 9.4 MB/s  0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


In [2]:
import json, torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("google/embeddinggemma-2",
                            model_kwargs={"torch_dtype": torch.float32})
ham_veri = json.load(open("veri.json", encoding="utf-8"))
kayitlar = ham_veri["kartlar"] if isinstance(ham_veri, dict) else ham_veri

emb = model.encode([k["metin"] for k in kayitlar], prompt_name="Document")
benzerlik = model.similarity(emb, emb)

sonuc = {}
for i, k in enumerate(kayitlar):
    skor = benzerlik[i].clone()
    skor[i] = -1  # kaydın kendisini atla
    en_iyi = torch.topk(skor, k=min(3, len(kayitlar)-1)).indices.tolist()
    sonuc[k["id"]] = [kayitlar[j]["id"] for j in en_iyi]

json.dump(sonuc, open("benzerler.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("Tamamlandı! Çıktı:", sonuc)

modules.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/22.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.49GB            

... (7 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

Tamamlandı! Çıktı: {'i01': ['i02', 'i03', 'i04'], 'i02': ['i01', 'i04', 'i03'], 'i03': ['i01', 'i02', 'i04'], 'i04': ['i02', 'i01', 'i03']}
